# Exploring the Population

This notebook demonstrates how to query and explore the full study population after ingestion and sequence calculation are complete.

**Prerequisites:** Run `01_getting_started.ipynb` first to create the `covid_35k.sqlite3` database.

**What you will learn:**
1. Listing and inspecting patients
2. Exploring individual patients via `PatientInstance`
3. Querying population-level sequences
4. Querying and filtering population-level frequencies

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import tspmdb
import pandas as pd

DB_PATH = '../covid_35k.sqlite3'
db = tspmdb.TspmDB(DB_PATH)
print('Opened database:', DB_PATH)

## 2. Querying Patients

The `db.population.patients()` method supports three return formats:
- **Default** — a list of `PatientInstance` objects (rich, per-patient API)
- **`as_list=True`** — a simple list of `patient_id` strings
- **`as_pandas=True`** — a Pandas DataFrame

In [ ]:
# Get a simple list of patient ID strings
patient_ids = db.population.patients(as_list=True)
print(f'Total patients: {len(patient_ids):,}')
print('First 5 patient IDs:', patient_ids[:5])

In [ ]:
# Get a DataFrame with both the string ID and the internal database integer key
patients_df = db.population.patients(as_pandas=True, with_ids=True)
patients_df.head()

## 3. Inspecting an Individual Patient

`db.population.patients()` (default mode) returns a list of `PatientInstance` objects.
Each instance exposes `.id`, `.events()`, and `.sequences()` for that specific patient.

In [ ]:
# Get PatientInstance objects
patients = db.population.patients()
patient = patients[0]

print('Patient ID:', patient.id)

In [ ]:
# View all observation events for this patient
events_df = patient.events(as_pandas=True)
print(f'Total events for {patient.id}: {len(events_df)}')
events_df.head(10)

In [ ]:
# View the transitive sequences derived from this patient's records
seqs_df = patient.sequences(as_pandas=True)
print(f'Total sequences for {patient.id}: {len(seqs_df)}')
seqs_df.sort_values('time_diff').head(10)

## 4. Population-Level Sequences

`db.population.sequences()` returns all sequences for all patients in a single query.
For large datasets, use `as_iterator=True` to avoid loading everything into memory at once.

In [ ]:
# Load all sequences as a DataFrame
sequences_df = db.population.sequences(as_pandas=True)
print(f'Total sequence records: {len(sequences_df):,}')
sequences_df.head()

In [ ]:
# Basic statistics on temporal distances
sequences_df['time_diff'].describe()

In [ ]:
# Memory-efficient iteration over sequences (useful for very large populations)
count = 0
for seq in db.population.sequences(as_iterator=True):
    count += 1
print(f'Counted {count:,} sequence records via iterator')

## 5. Population-Level Frequencies

`db.population.frequencies()` reads from the pre-calculated `frequencies` table.
It supports filtering by `observation1` (obs_code_1) and/or `observation2` (obs_code_2).

- Pass a **single string** to match one code.
- Pass a **list of strings** to match any of them (OR logic).
- Filters are **AND**-ed across `observation1` and `observation2`.
- Invalid codes raise a **`KeyError`** immediately.

In [ ]:
# All frequencies — sorted by patient_cnt descending to see the most common sequences
freq_df = db.population.frequencies(as_pandas=True)
print(f'Total frequency rows: {len(freq_df):,}')
freq_df.sort_values('patient_cnt', ascending=False).head(10)

In [ ]:
# Pick the most common obs_code_1 from the results above and filter by it
top_code = freq_df.sort_values('patient_cnt', ascending=False).iloc[0]['obs_code_1']
print('Filtering by obs_code_1:', top_code)

filtered_df = db.population.frequencies(observation1=top_code, as_pandas=True)
filtered_df.sort_values('patient_cnt', ascending=False).head(10)

In [ ]:
# Filter by multiple obs_code_1 values (OR logic within the list)
top_codes = freq_df.sort_values('patient_cnt', ascending=False)['obs_code_1'].unique()[:3].tolist()
print('Filtering by obs_code_1 IN:', top_codes)

multi_df = db.population.frequencies(observation1=top_codes, as_pandas=True)
print(f'Rows returned: {len(multi_df):,}')
multi_df.head()

In [ ]:
# Filter by both observation1 AND observation2
top_code_2 = freq_df.sort_values('patient_cnt', ascending=False).iloc[0]['obs_code_2']

both_df = db.population.frequencies(
    observation1=top_code,
    observation2=top_code_2,
    as_pandas=True
)
print(f'Rows for ({top_code} -> {top_code_2}): {len(both_df)}')
both_df

In [ ]:
# Demonstrate KeyError for invalid observation codes
try:
    db.population.frequencies(observation1='INVALID_CODE_XYZ')
except KeyError as e:
    print('KeyError raised as expected:', e)

In [ ]:
# Return raw integer IDs instead of string codes (advanced usage)
raw_df = db.population.frequencies(observation1=top_code, with_ids=True, as_pandas=True)
raw_df.head()

## 6. Close the Database

In [ ]:
db.close()
print('Database closed.')

## Next Steps

- **[03_subpopulations.ipynb](03_subpopulations.ipynb)** — Define patient cohorts (e.g., by diagnosis) and compare their sequence frequency profiles.